# Agent Harness Engineering · Data Figures
### The four real matplotlib figures behind a 34-episode architecture course

*Agent Harness Engineering* teaches how to build the scaffolding around an LLM that turns it into an autonomous coding agent — the loop, context, tools/MCP, memory, planning, security, evaluation, and a from-scratch capstone. Most of the course is **architecture diagrams**, but four episodes carry a real data figure. This notebook reproduces those four, seeded and to the pixel:

- **HE07** — real softmax attention over a short sentence (a heatmap).
- **HE08** — a context-window budget breakdown (what fills the finite window).
- **HE22** — the vector-index recall-vs-speed trade-off (flat / IVF / HNSW).
- **HE28** — execution-based eval: the FAIL_TO_PASS / PASS_TO_PASS grid that defines *Resolved*.

Run the setup cell once, then each episode writes its program to disk, runs it, and shows the figure.

In [ ]:
!pip install -q numpy matplotlib

> Each episode below: **write** the program → **run** it → **show** the figure it saved. The code is the real generator used to produce the on-screen figure in the video.

## HE07 · LLM Internals I · Tokens, Embeddings & Attention

A language model can't read English. It reads numbers. So before any thinking happens, your sentence is chopped into chunks called tokens — roughly word-pieces — and each token is looked up in a giant table that turns it into a long list of numbers called a vector. Those vectors live in a space where meaning *is* distance: "king" and "queen" land near each other, "banana" lands far away. Then comes the engine that made modern AI possible — attention — where every token in your prompt gets to glance at every other token and weigh how much each one matters to it. That all-looks-at-all step is incredibly powerful, but it has a brutal cost: if you double the number of tokens, the work doesn't double, it *quadruples*. Hold onto that number squared — it is the quiet reason a harness fights so hard over every token.

In [ ]:
%%writefile he07_attention.py
"""HE07 · attention heatmap for the hero sentence (seeded, illustrative).
Tokens treated 1-per-word for a clean N=10 demonstration matrix.
Saves remotion/public/harnessfig/he07.png — a real softmax attention heatmap.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(7)
TOKENS = ["the", "cat", "sat", "on", "the", "mat", "because", "it", "was", "tired"]
N = len(TOKENS)  # 10

# build illustrative pre-softmax scores: mild base + self + a few real syntactic links
S = np.random.rand(N, N) * 0.6
S += np.eye(N) * 1.3                 # self-attention
def link(r, c, w):                  # row token attends to col token
    S[r, c] += w
link(7, 1, 3.4)   # "it"   -> "cat"   (the famous coreference)
link(2, 1, 2.0)   # "sat"  -> "cat"   (subject)
link(5, 3, 1.4)   # "mat"  -> "on"
link(5, 4, 1.2)   # "mat"  -> "the"
link(9, 7, 1.6)   # "tired"-> "it"
link(9, 1, 1.3)   # "tired"-> "cat"
link(6, 2, 1.1)   # "because" -> "sat"

# row-wise softmax -> attention weights (each row sums to 1)
A = np.exp(S - S.max(axis=1, keepdims=True))
A = A / A.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.4, 6.6), dpi=100)
fig.patch.set_facecolor("#ffffff")
im = ax.imshow(A, cmap="Blues", vmin=0, vmax=A.max())
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(TOKENS, rotation=45, ha="right", fontsize=11, family="monospace")
ax.set_yticklabels(TOKENS, fontsize=11, family="monospace")
ax.set_xlabel("attends to (key)", fontsize=12)
ax.set_ylabel("query token", fontsize=12)
ax.set_title("attention weights  ·  N = 10 tokens  ·  10 x 10 = 100 cells", fontsize=12, pad=12)
# annotate the strongest off-diagonal cell ("it" -> "cat")
ax.text(1, 7, f"{A[7,1]:.2f}", ha="center", va="center", fontsize=10,
        color="white", weight="bold")
ax.add_patch(plt.Rectangle((0.5, 6.5), 1, 1, fill=False, edgecolor="#ffd166", lw=2.4))
ax.text(2.2, 7, '"it" -> "cat"', fontsize=10, color="#b8860b", va="center")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("weight (row sums to 1)", fontsize=10)
ax.set_xticks(np.arange(-.5, N, 1), minor=True)
ax.set_yticks(np.arange(-.5, N, 1), minor=True)
ax.grid(which="minor", color="#dfe3ea", linewidth=0.8)
ax.tick_params(which="minor", length=0)
fig.tight_layout()

out = Path("he07.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, facecolor="#ffffff", bbox_inches="tight")
print("saved", out)
print("N =", N)
print('it->cat weight =', round(float(A[7,1]), 3))
print('it row top-3:', sorted(((round(float(A[7,j]),3), TOKENS[j]) for j in range(N)), reverse=True)[:3])
print('diag mean =', round(float(np.mean(np.diag(A))), 3))
print('cost: N^2 =', N*N, '| 1000^2 =', 1000*1000)


In [ ]:
!python he07_attention.py

![HE07 figure](he07.png)

## HE08 · LLM Internals II · Context Windows & Inference

A context window is not a bucket you fill to the brim — it's a desk. A small desk where the model has to keep eye contact with every paper at once (that's the n-squared attention from last episode). Pile on too many papers and the model doesn't just run out of room; it starts losing track *before* the desk is full, because its attention is stretched thinner across everything. Running the model has two moves: first it reads the whole desk in one big gulp (prefill), then it writes its answer one word at a time (decode). The clever harness trick is to notice that the *front* of the desk barely changes turn to turn — the same system prompt, the same rules — so why re-read it every time? Cache that work. Keep the unchanging stuff first and never touch it, and you stop paying the bulk-read tax over and over.

In [ ]:
%%writefile he08_context.py
"""HE08 · context-window fuel-gauge + recall-vs-tokens 'Dumb Zone' curve (illustrative).
Left: a vertical stacked gauge (static / history / hard limit).
Right: recall quality vs tokens used, sagging well before the hard limit (the Dumb Zone).
The ~167k compaction marker is ILLUSTRATIVE (exact trigger is unofficial; concept = intervene early).
Saves remotion/public/harnessfig/he08.png.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path

np.random.seed(8)
LIMIT = 200_000          # Claude 200k context window (well-known)
STATIC = 15_000          # system + tools + rules (cacheable prefix)
HISTORY_TOP = 180_000    # current fill of conversation history
COMPACT = 167_000        # illustrative early-compaction marker (Claude Code)

def recall(t):           # quality of finding a fact at usage t (sigmoid sag)
    return 0.95 - 0.40 / (1.0 + np.exp(-(t - 110_000) / 22_000))

t = np.linspace(0, LIMIT, 400)
r = recall(t)
DUMB_ONSET = t[np.argmax(r < 0.85)]   # where recall first dips below 0.85

fig = plt.figure(figsize=(6.4, 6.8), dpi=100)
fig.patch.set_facecolor("#ffffff")
gs = GridSpec(1, 2, width_ratios=[1, 2.4], wspace=0.05)
axg = fig.add_subplot(gs[0]); axr = fig.add_subplot(gs[1], sharey=axg)

# ---- left: vertical fuel gauge ----
W = 0.6
axg.bar(0, STATIC, width=W, bottom=0, color="#ffd166", label="static (cached)")
axg.bar(0, HISTORY_TOP - STATIC, width=W, bottom=STATIC, color="#9fb4cf", label="history (grows)")
axg.bar(0, LIMIT - HISTORY_TOP, width=W, bottom=HISTORY_TOP, color="#e9edf3")
axg.axhline(LIMIT, color="#ff6b6b", lw=2.4)
axg.text(0, LIMIT + 4000, "HARD LIMIT 200k", ha="center", color="#ff6b6b", fontsize=10, weight="bold")
axg.text(0, STATIC / 2, "static\n15k", ha="center", va="center", fontsize=8, family="monospace")
axg.text(0, (STATIC + HISTORY_TOP) / 2, "history", ha="center", va="center", fontsize=9, family="monospace")
axg.set_xlim(-0.5, 0.5); axg.set_xticks([]); axg.set_title("window", fontsize=11)
axg.set_ylabel("tokens used", fontsize=12)

# ---- right: recall vs tokens (Dumb Zone) ----
axr.fill_betweenx(t, 0, r, where=(t >= DUMB_ONSET), color="#ffd166", alpha=0.18)
axr.plot(r, t, color="#4aa3ff", lw=3)
axr.axhline(LIMIT, color="#ff6b6b", lw=2.0)
axr.axhline(COMPACT, color="#5ce1c4", lw=2.0, ls="--")
axr.text(0.02, COMPACT + 4000, "~167k  compact early (Claude Code)", color="#138a72", fontsize=9.5)
axr.text(0.02, DUMB_ONSET + 30000, "THE DUMB ZONE", color="#b8860b", fontsize=11, weight="bold")
axr.text(0.02, DUMB_ONSET + 12000, "recall sags before the limit", color="#b8860b", fontsize=9)
axr.annotate("flat + high", xy=(recall(20000), 20000), xytext=(0.35, 35000),
             fontsize=9, color="#2b6cb0", arrowprops=dict(arrowstyle="->", color="#2b6cb0"))
axr.set_xlim(0, 1.0); axr.set_xlabel("recall quality", fontsize=12)
axr.set_title("recall vs context used", fontsize=11)
axr.invert_xaxis()  # high recall on the left edge near low tokens reads naturally
plt.setp(axr.get_yticklabels(), visible=False)

axg.set_ylim(0, LIMIT + 12000)
axg.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
fig.suptitle("context window: a finite attention budget, not a bucket", fontsize=12.5, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])

out = Path("he08.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, facecolor="#ffffff", bbox_inches="tight")
print("saved", out)
print("LIMIT =", LIMIT, "| STATIC =", STATIC, "| COMPACT =", COMPACT)
print("recall@0 =", round(float(recall(0)), 3), "| recall@110k =", round(float(recall(110_000)), 3), "| recall@200k =", round(float(recall(LIMIT)), 3))
print("Dumb Zone onset ~", int(DUMB_ONSET // 1000), "k tokens (recall<0.85), well left of 200k limit")


In [ ]:
!python he08_context.py

![HE08 figure](he08.png)

## HE22 · Vector Databases and Embeddings

An embedding turns meaning into a location. The model has learned a giant coordinate space where "cat" and "kitten" land near each other and "tax form" lands far away — so "find similar meaning" becomes the much simpler "find nearby points." A vector database is the machine that does that "find nearby points" search fast. The catch: checking every point against your query is exact but unbearably slow at millions of vectors. So every vector DB cheats with approximate nearest-neighbor search, and every cheat is a deal — you give up a little recall to buy speed or memory. It's exactly like organizing a library: shelve nothing and you find every book by walking every aisle (perfect, slow); shelve by section and you check three sections instead of the whole building (fast, but you might miss a misfiled book).

In [ ]:
%%writefile he22_vectordb.py
"""HE22 · Vector DB ANN trade-off — recall vs speed, bubble size = memory (RAM).
Four index structures (Flat / IVF / HNSW / PQ) plotted as one point each in the
recall-vs-speed plane; bubble area encodes the third axis (memory). The whole point:
no index sits top-right AND tiny — you can be great at any two of recall/speed/memory,
never all three. Numbers are ILLUSTRATIVE (typical ANN-benchmark behavior, not a single run).
Saves remotion/public/harnessfig/he22.png.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(22)

# name, recall@10, queries/sec, RAM (relative units -> bubble area), color
IDX = [
    ("Flat",  1.000,    80, 55, "#4aa3ff"),   # brute force: perfect recall, slowest, full vectors
    ("IVF",   0.930,  2600, 32, "#9fd97a"),   # buckets: cheap memory, recall depends on probes
    ("HNSW",  0.985,  6500, 98, "#bd7dff"),   # graph: best speed/recall, RAM-hungry
    ("PQ",    0.740,  9500, 10, "#ff6ec7"),   # compressed: billions fit in RAM, recall cost
]

fig, ax = plt.subplots(figsize=(6.2, 7.4), dpi=100)
fig.patch.set_facecolor("#ffffff")
ax.set_facecolor("#ffffff")

for name, rec, qps, ram, col in IDX:
    ax.scatter(rec, qps, s=ram * 46, c=col, alpha=0.45,
               edgecolors=col, linewidths=2.4, zorder=3)
    ax.annotate(name, (rec, qps), xytext=(0, 14), textcoords="offset points",
                ha="center", color=col, fontsize=15, weight="bold")

ax.set_yscale("log")
ax.set_xlim(0.66, 1.05)
ax.set_ylim(45, 20000)
ax.set_xlabel("recall@10   (did you find the true neighbors)", fontsize=11)
ax.set_ylabel("queries / sec   (speed, log scale)", fontsize=11)
ax.set_title("ANN trade-off triangle, as a plot\nbubble size = memory (RAM)",
             fontsize=14, weight="bold", pad=14)

# the impossible corner: fast + exact + tiny RAM (no real index reaches it)
ax.scatter(1.01, 15000, s=95, marker="x", c="#ff6b6b", zorder=5, linewidths=2.6)
ax.annotate("impossible corner:\nfast + exact + tiny RAM",
            (1.01, 15000), xytext=(-16, 0), textcoords="offset points",
            ha="right", va="center", fontsize=9.5, color="#ff6b6b", style="italic")

# color-keyed legend in the empty lower-left (the third axis = bubble = memory)
legend = [
    ("Flat", "perfect recall, slowest", "#4aa3ff"),
    ("IVF",  "cheap RAM, recall via probes", "#9fd97a"),
    ("HNSW", "fast + accurate, big RAM", "#bd7dff"),
    ("PQ",   "tiny RAM, lower recall", "#ff6ec7"),
]
ly = [320, 215, 145, 98]
for (nm, desc, col), yy in zip(legend, ly):
    ax.text(0.675, yy, nm, color=col, fontsize=11, weight="bold",
            family="monospace", va="center")
    ax.text(0.74, yy, desc, color="#5b6472", fontsize=9,
            family="monospace", va="center")

ax.grid(True, which="both", color="#e6e9ef", lw=0.8, zorder=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.tick_params(labelsize=9)

out = Path("he22.png")
fig.tight_layout()
fig.savefig(out, dpi=100, facecolor="#ffffff", bbox_inches="tight")
print("saved", out)
for name, rec, qps, ram, col in IDX:
    print(f"  {name:5s} recall={rec:.3f}  qps={qps:5d}  ram={ram}")


In [ ]:
!python he22_vectordb.py

![HE22 figure](he22.png)

## HE28 · Evaluation & Benchmarking

An agent's answer isn't a single string you can diff against a gold answer — it's
a *trajectory*: a multi-step path of reasoning, tool calls, retries, and
hand-offs. So evaluation has to grade both the destination *and* the route, like
a math exam where the final answer and the shown work both matter. The cleanest
grader needs no judgment at all: if you can run real tests, you let the tests
decide. SWE-bench does exactly this — apply the agent's patch, run the repo's own
test suite, and check that the broken tests now pass and the working ones still
pass. When you can't run tests, you fall back to an LLM acting as a teaching
assistant grading essays — fast and scalable, but you have to spot-check its
grades against your own or grade inflation creeps in.

In [ ]:
%%writefile he28_swebench.py
"""HE28 · SWE-bench execution grader — before/after test-status grid.
The grader runs the repo's own tests before and after applying the agent's patch.
FAIL_TO_PASS tests must flip red->green (the fix works); PASS_TO_PASS tests must
stay green (no regression). "Resolved" = ALL FAIL_TO_PASS pass AND zero
PASS_TO_PASS break. This is execution-based grading: no opinion, just code.
Conceptual illustration (not a benchmark run). Saves remotion/public/harnessfig/he28.png.
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

GREEN = "#4caf6a"
RED = "#e5534b"

# rows top->bottom; 1 = pass (green), 0 = fail (red)
rows = [
    ("FAIL_TO_PASS  #1", 0, 1),
    ("FAIL_TO_PASS  #2", 0, 1),
    ("FAIL_TO_PASS  #3", 0, 1),
    ("PASS_TO_PASS  #1", 1, 1),
    ("PASS_TO_PASS  #2", 1, 1),
    ("PASS_TO_PASS  #3", 1, 1),
]
n = len(rows)

fig, ax = plt.subplots(figsize=(6.4, 7.2), dpi=100)
fig.patch.set_facecolor("#ffffff")
ax.set_facecolor("#ffffff")

cw, ch, gap = 1.0, 0.74, 0.16
x_before, x_after = 1.9, 3.2

for i, (label, b, a) in enumerate(rows):
    y = (n - 1 - i) * (ch + gap)
    ax.text(1.78, y + ch / 2, label, ha="right", va="center",
            fontsize=12, family="monospace", color="#2a2f37")
    for x, v in ((x_before, b), (x_after, a)):
        col = GREEN if v else RED
        ax.add_patch(Rectangle((x, y), cw, ch, facecolor=col, alpha=0.55,
                               edgecolor=col, linewidth=2))
        ax.text(x + cw / 2, y + ch / 2, "pass" if v else "fail",
                ha="center", va="center", fontsize=11, family="monospace",
                color="#1c3d28" if v else "#5c1f1c", weight="bold")

top = n * (ch + gap)
ax.text(x_before + cw / 2, top + 0.05, "BEFORE\npatch", ha="center", va="bottom",
        fontsize=12, weight="bold", color="#5b6472")
ax.text(x_after + cw / 2, top + 0.05, "AFTER\npatch", ha="center", va="bottom",
        fontsize=12, weight="bold", color="#5b6472")

# arrow hint between columns on the FAIL_TO_PASS rows (the flip)
for i in range(3):
    y = (n - 1 - i) * (ch + gap) + ch / 2
    ax.annotate("", xy=(x_after - 0.06, y), xytext=(x_before + cw + 0.06, y),
                arrowprops=dict(arrowstyle="->", color="#4caf6a", lw=1.8))

ax.set_xlim(-0.2, x_after + cw + 0.4)
ax.set_ylim(-1.5, top + 1.0)
ax.axis("off")
ax.set_title("SWE-bench grader: run the repo's own tests",
             fontsize=14, weight="bold", color="#222", pad=12)

ax.text((x_before + x_after + cw) / 2, -0.55,
        "Resolved = every FAIL_TO_PASS flips to pass  AND",
        ha="center", fontsize=11, family="monospace", color="#1c3d28")
ax.text((x_before + x_after + cw) / 2, -0.95,
        "every PASS_TO_PASS stays pass  (no regression)",
        ha="center", fontsize=11, family="monospace", color="#1c3d28")
ax.text((x_before + x_after + cw) / 2, -1.35,
        "execution-based: no opinion, just code",
        ha="center", fontsize=10, style="italic", color="#7a828c")

out = Path("he28.png")
fig.tight_layout()
fig.savefig(out, dpi=100, facecolor="#ffffff", bbox_inches="tight")
print("saved", out)
print("FAIL_TO_PASS: before=fail after=pass (must flip)")
print("PASS_TO_PASS: before=pass after=pass (must hold)")
print("Resolved = all F2P green AND all P2P green")


In [ ]:
!python he28_swebench.py

![HE28 figure](he28.png)

---
*Built with [Claude Code](https://claude.com/claude-code). Architecture-first · build a coding agent from scratch · @kader_mohideen*